# 3D brain bleed (and lesion) segmentation with 3D/VR output (Colab, `3d` branch)

**Before you start:** `Runtime → Change runtime type → T4 GPU` (A100 if you have Colab Pro), then run the cells in order.
Checkpoints and results go to Google Drive, so a disconnect does not lose the training: re-run from step 1 and the training cell resumes.

Research use only, not for diagnosis.

## 1. Settings

In [ ]:
TASK  = "bhsd"        # "bhsd" (haemorrhage, CT) | "isles22" (ischaemic stroke, MRI) | "brats" (tumour, MRI)
MODEL = "segresnet"   # "segresnet" | "swinunetr" | "dynunet"

DS = {"brats": ("Dataset501_BraTS", 501), "isles22": ("Dataset502_ISLES22", 502), "bhsd": ("Dataset503_BHSD", 503)}[TASK]
PRESET = {"brats": "brats", "isles22": "stroke", "bhsd": "bhsd"}[TASK]
!nvidia-smi -L

## 2. Google Drive (stores downloads, checkpoints and results)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
BASE = "/content/drive/MyDrive/brain-clot-3d"
OUT = f"{BASE}/runs/{TASK}_{MODEL}"
os.makedirs(OUT, exist_ok=True); os.makedirs(f"{BASE}/downloads", exist_ok=True)
print("Results ->", OUT)

## 3. Get the code
Works without a token if the repo is public. If it is private, add a `GITHUB_TOKEN` secret (key icon, read-only Contents) as for the 2D notebook.

In [ ]:
import subprocess, shutil
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@github.com/ChanduVenkatSaiNarne/Brain-Clot-Edema-midlineshift.git" if token else "https://github.com/ChanduVenkatSaiNarne/Brain-Clot-Edema-midlineshift.git"
REPO = "/content/Brain-Clot-Edema-midlineshift-3d"
if not os.path.exists(f"{REPO}/.git"):
    shutil.rmtree(REPO, ignore_errors=True)
    r = subprocess.run(["git", "clone", "-q", url, REPO], capture_output=True, text=True)
    print((r.stderr.replace(token, "***") if token else r.stderr) or "Cloned.")
%cd /content/Brain-Clot-Edema-midlineshift-3d
!git fetch -q && git checkout -q 3d && git reset -q --hard origin/3d
!git log --oneline -1
assert os.path.exists("train_3d.py"), "Code download failed"
!pip -q install -r requirements.txt

## 4. Get the data (downloaded once, cached in Drive)

- **isles22**: ISLES 2022 from Zenodo (public).
- **bhsd**: BHSD `label_192.zip` from Hugging Face (public).
- **brats**: register on Synapse, download the BraTS training zip, put it in Drive as `MyDrive/brain-3d/downloads/brats.zip`.

In [ ]:
import glob
URLS = {"isles22": "https://zenodo.org/records/7153326/files/ISLES-2022.zip?download=1",
        "bhsd": "https://huggingface.co/datasets/WuBiao/BHSD/resolve/main/label_192.zip"}
ZIP = f"{BASE}/downloads/{TASK}.zip"
if not os.path.exists(ZIP) or os.path.getsize(ZIP) < 1e6:
    if TASK == "brats":
        raise SystemExit("Upload the BraTS training zip to " + ZIP)
    !wget -q --show-progress -O "{ZIP}" "{URLS[TASK]}"
SRC = f"/content/data/{TASK}"
os.makedirs(SRC, exist_ok=True)
!unzip -q -o "{ZIP}" -d {SRC}
for z in glob.glob(f"{SRC}/**/*.zip", recursive=True):     # nested zips
    !unzip -q -o "{z}" -d "{os.path.dirname(z)}"
!du -sh {SRC}; find {SRC} -maxdepth 3 -type d | head -20

## 5. Convert to the standard (nnU-Net) format

In [ ]:
RAW = "/content/nnUNet_raw"
if TASK == "brats":
    !python to_nnunet.py brats --src {SRC} --out {RAW} --id 501
elif TASK == "isles22":
    !python to_nnunet.py isles22 --src {SRC} --out {RAW} --id 502
else:
    !python to_nnunet.py pairs --src {SRC} --out {RAW} --id 503 --name BHSD --channel CT --label-names bleed_1,bleed_2,bleed_3,bleed_4,bleed_5
DATASET = f"{RAW}/{DS[0]}"
!cat {DATASET}/dataset.json

## 6. Quick test (12 cases, 2 epochs)

In [ ]:
!python train_3d.py --dataset {DATASET} --model {MODEL} --out /content/test_run --max-cases 12 --epochs 2 --val-every 1

## 7. Full training (hours; resumes automatically after a disconnect)

In [ ]:
!python train_3d.py --dataset {DATASET} --model {MODEL} --out {OUT} --resume

## 8. Test results and curves

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
r = json.load(open(f"{OUT}/test_results.json")); print(json.dumps(r, indent=2))
h = pd.read_csv(f"{OUT}/history.csv")
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(h.epoch, h.loss); ax[0].set_title("Training loss")
v = h.dropna(subset=["val_dice_mean"])
for c in [c for c in h.columns if c.startswith("val_dice")]: ax[1].plot(v.epoch, v[c], label=c)
ax[1].set_title("Validation Dice"); ax[1].legend(); plt.show()

## 9. Predict a test case and export a 3D model for VR

In [ ]:
from google.colab import files
case = json.load(open(f"{OUT}/split.json"))["test"][0]
pred = f"{OUT}/pred_{case}.nii.gz"
!python predict_3d.py --ckpt {OUT}/best.pt --dataset {DATASET} --case {case} --out {pred}
ctx = f"{DATASET}/imagesTr/{case}_0000.nii.gz"
mode = "ct-skull" if TASK == "bhsd" else "mri"
!python export_mesh.py --labels {pred} --preset {PRESET} --context {ctx} --context-mode {mode} --out {OUT}/{case}.glb
# ground truth for comparison
!python export_mesh.py --labels {DATASET}/labelsTr/{case}.nii.gz --preset {PRESET} --context {ctx} --context-mode {mode} --out {OUT}/{case}_truth.glb
files.download(f"{OUT}/{case}.glb")

Open the `.glb` in the viewer (`viewer/index.html` on GitHub Pages: **Open .glb**), in the Quest browser for VR, or in 3D Slicer.
For the BraTS preset the tumour label map is shown as edema (1), necrotic core (2) and enhancing tumour (3).

## 10. (Optional) nnU-Net baseline on the same data

In [ ]:
!pip -q install nnunetv2
os.environ.update(nnUNet_raw=RAW, nnUNet_preprocessed="/content/nnUNet_preprocessed", nnUNet_results=f"{BASE}/nnUNet_results")
!nnUNetv2_plan_and_preprocess -d {DS[1]} --verify_dataset_integrity
!nnUNetv2_train {DS[1]} 3d_fullres 0